In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from nltk.tokenize import TweetTokenizer
import nltk
from tqdm import tqdm

In [ ]:
# 缩写转换字典
ABBREVIATIONS = {
   "he's": "he is", 
   "there's": "there is", 
   "We're": "We are", 
   "That's": "That is", 
   "won't": "will not", 
   "they're": "they are", 
   "Can't": "Can not", 
   "wasn't": "was not", 
   "don't": "do not", 
   "aren't": "are not", 
   "isn't": "is not", 
   "What's": "What is", 
   "haven't": "have not", 
   "hasn't": "has not", 
   "There's": "There is", 
   "He's": "He is", 
   "It's": "It is", 
   "You're": "You are", 
   "I'M":"I am", 
   "shouldn't": "should not", 
   "wouldn't": "would not", 
   "i'm": "I am", 
   "I'm": "I am", 
   "Isn't": "is not", 
   "Here's":"Here is", 
   "you've": "you have", 
   "we're": "we are", 
   "what's":"what is", 
   "couldn't": "could not", 
   "we've": "we have", 
   "it's": "it is", 
   "doesn't": "does not", 
   "It's": "It is", 
   "who's": "who is", 
   "I've": "I have", 
   "y'all": "you all", 
   "would've": "would have", 
   "it'll": "it will", 
   "we'll": "we will", 
   "We've": "We have", 
   "he'll": "he will", 
   "Y'all": "You all", 
   "Weren't": "Were not", 
   "Didn't": "Did not", 
   "they'll": "they will", 
   "they'd": "they would", 
   "DON'T": "DO NOT", 
   "That's": "That is", 
   "they've": "they have", 
   "i'd": "I would", 
   "should've": "should have", 
   "You're": "You are", 
   "where's": "where is",
   "we'd": "we would", 
   "i'll": "I will", 
   "weren't": "were not", 
   "They're": "They are", 
   "you'll": "you will", 
   "I'd": "I would", 
   "let's": "let us", 
   "it's": "it is", 
   "can't": "can not", 
   "don't": "do not", 
   "you're": "you are", 
   "i've": "I have", 
   "that's": "that is", 
   "i'll": "I will", 
   "doesn't": "does not", 
   "i'd": "I would", 
   "didn't": "did not", 
   "ain't": "am not", 
   "I've": "I have", 
   "Don't": "do not", 
   "I'll": "I will", 
   "I'd":"I would", 
   "Let's": "Let us", 
   "you'd": "You would", 
   "It's": "It is", 
   "Ain't": "am not", 
   "Haven't": "Have not", 
   "Could've": "Could have", 
   "youve": "you have",   
   "w/": "with",
    "w/o": "without",
    "b/c": "because",
    "thx": "thanks",
    "u": "you",
    "r": "are",
    "rt": "retweet"
}

In [ ]:
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet


class CustomTweetCleaner:
    def __init__(self):
        self.tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)
        self.lemmatizer = WordNetLemmatizer()

    def get_wordnet_pos(self, treebank_tag):
        """将NLTK的POS标签转换为WordNet的POS标签"""
        if treebank_tag.startswith('J'):
            return wordnet.ADJ
        elif treebank_tag.startswith('V'):
            return wordnet.VERB
        elif treebank_tag.startswith('N'):
            return wordnet.NOUN
        elif treebank_tag.startswith('R'):
            return wordnet.ADV
        else:
            return wordnet.NOUN  # 默认使用名词

    def clean_text(self, text):
        """清洗单条文本并进行词形还原"""
        if not isinstance(text, str):
            text = str(text)
        # 移除URL
        text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
        # 替换缩写
        for abbr, full in ABBREVIATIONS.items():
            text = re.sub(r'\b' + re.escape(abbr) + r'\b', full, text, flags=re.IGNORECASE)
        # 移除标点符号
        text = re.sub(r'[^\w\s]', '', text)
        # 分词并进行词性标注
        tokens = self.tokenizer.tokenize(text)
        tagged_tokens = pos_tag(tokens)
        # 词形还原
        lemmatized_tokens = [
            self.lemmatizer.lemmatize(token, self.get_wordnet_pos(pos))
            for token, pos in tagged_tokens
        ]
        # 合并还原后的词并清理多余空格
        text = ' '.join(lemmatized_tokens).strip()
        return text

    def tokenize(self, text):
        """自定义分词器"""
        cleaned_text = self.clean_text(text)
        tokens = self.tokenizer.tokenize(cleaned_text)
        return tokens

使用bert

In [ ]:
import pandas as pd
import re
import numpy as np
import torch
print(torch.__version__)
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split,StratifiedKFold
from nltk.tokenize import TweetTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag
import optuna
import nltk
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import get_linear_schedule_with_warmup

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, keywords, locations, labels, tokenizer, max_len=128):
        self.texts = texts
        self.keywords = keywords
        self.locations = locations
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.cleaner = CustomTweetCleaner()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        keyword = self.keywords[idx] if self.keywords[idx] else 'none'
        location = self.locations[idx] if self.locations[idx] else 'no_location'
        label = self.labels[idx]

        cleaned_text = self.cleaner.clean_text(text)
        cleaned_keyword = self.cleaner.clean_text(keyword)
        cleaned_location = self.cleaner.clean_text(location)

        # 拼接 location, keyword, text
        input_text = f"{cleaned_location} [SEP] {cleaned_keyword} [SEP] {cleaned_text}"

        encoding = self.tokenizer.encode_plus(
            input_text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
def plot_classification_metrics(epochs, val_accuracies, val_f1_scores, val_precisions, val_recalls,batch_size, learning_rate, fold=None):
    """绘制分类报告曲线：Accuracy, F1, Precision, Recall"""
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, epochs + 1), val_accuracies, label='Val Accuracy', color='blue', marker='o')
    plt.plot(range(1, epochs + 1), val_f1_scores, label='Val F1 Score', color='gold', marker='o')
    plt.plot(range(1, epochs + 1), val_precisions, label='Val Precision', color='cyan', marker='o')
    plt.plot(range(1, epochs + 1), val_recalls, label='Val Recall', color='purple', marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('Metric Value')
    lr_str = f"{learning_rate:.0e}"  # 科学计数法表示
    if fold is not None:
        plt.title(f'Fold {fold} Metrics (bs={batch_size}, lr={lr_str})')
        # filename = f'picture/classification_metrics_fold_{fold}_bs{batch_size}_lr{lr_str}.png'
    else:
        plt.title(f'CV Metrics (bs={batch_size}, lr={lr_str})')
        # filename = f'picture/classification_metrics_cv_bs{batch_size}_lr{lr_str}.png'
    plt.legend()
    plt.grid(True)
    plt.ylim(0, 1)
    plt.xticks(range(1, epochs + 1))
    # plt.savefig(filename)
    plt.show()
    plt.close()

def plot_loss_curves(epochs, train_losses, val_losses,batch_size, learning_rate, fold=None):
    """绘制损耗曲线：Train Loss, Val Loss"""
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, epochs + 1), train_losses, label='Train Loss', color='cyan', marker='o')
    plt.plot(range(1, epochs + 1), val_losses, label='Val Loss', color='red', marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    lr_str = f"{learning_rate:.0e}"
    if fold is not None:
        plt.title(f'Fold {fold} Loss (bs={batch_size}, lr={lr_str})')
        # filename = f'picture/loss_curves_fold_{fold}_bs{batch_size}_lr{lr_str}.png'
    else:
        plt.title(f'CV Loss (bs={batch_size}, lr={lr_str})')
        # filename = f'picture/loss_curves_cv_bs{batch_size}_lr{lr_str}.png'
    plt.legend()
    plt.grid(True)
    plt.ylim(0, max(max(train_losses), max(val_losses)) * 1.1)
    plt.xticks(range(1, epochs + 1))
    # plt.savefig(filename)
    plt.show()
    plt.close()

In [ ]:
def compute_class_weights(data):
    """Calculate class weights based on inverse class frequency."""
    class_counts = data['target'].value_counts()
    total_samples = len(data)
    class_weights = {}
    
    # Inverse frequency: weight = total_samples / (n_classes * class_count)
    for target in class_counts.index:
        class_weights[target] = total_samples / (2 * class_counts[target])
    
    # Convert to tensor for use in loss function
    weights = [class_weights[0], class_weights[1]]
    return torch.tensor(weights, dtype=torch.float)

In [ ]:
def train_final_bert_model(data, model_name, epochs=10, batch_size=16, max_len=128, learning_rate=2e-5):
    # cleaner = CustomTweetCleaner()
    # data['text_cleaned'] = data['text'].apply(cleaner.clean_text)
    # data['keyword'] = data['keyword'].fillna('none').apply(cleaner.clean_text)
    texts = data['text'].tolist()
    keywords = data['keyword'].tolist()
    locations = data['location'].tolist()
    targets = data['target'].tolist()

    

    # Compute class weights
    class_weights = compute_class_weights(data)
    print(f"Class weights: {class_weights}")

    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2, hidden_dropout_prob=0.2)
    for name, param in model.bert.named_parameters():
        if not (name.startswith("encoder.layer.9") or 
                name.startswith("encoder.layer.10") or 
                name.startswith("encoder.layer.11")):
            param.requires_grad = False
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    class_weights = class_weights.to(device)  # Move weights to device

    full_dataset = TweetDataset(texts, keywords, locations, targets, tokenizer, max_len)
    full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True)

    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    total_steps = len(full_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0.1 * total_steps, num_training_steps=total_steps)

    # Define weighted loss function
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

    avg_losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(full_loader, desc=f"Final Training Epoch {epoch + 1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, labels)  # Use weighted loss
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

        avg_loss = total_loss / len(full_loader)
        avg_losses.append(avg_loss)
        print(f"Epoch {epoch + 1} - Avg Training Loss: {avg_loss:.4f}")

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, epochs + 1), avg_losses, marker='o', color='red', label='Avg Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Final Model Training Loss Curve (bs={batch_size}, lr={learning_rate:.0e})')
    plt.grid(True)
    plt.xticks(range(1, epochs + 1))
    plt.legend()
    plt.tight_layout()
    # plt.savefig('picture/final_training_loss.png')
    plt.show()
    plt.close()

    return model, tokenizer

In [ ]:

LOCAL_BERT_PATH = r"/kaggle/input/bert_model/transformers/default/1/bert_model"
data=pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")

# LOCAL_BERT_PATH = r"C:\Users\25853\hubmodel\86b5e0934494bd15c9632b12f734a8a67f723594"
# data=pd.read_csv("../data/train.csv")


# 清洗训练数据
cleaner = CustomTweetCleaner()
data['text'] = data['text'].apply(cleaner.clean_text)
data['keyword'] = data['keyword'].fillna('none').apply(cleaner.clean_text)
data['location'] = data['location'].fillna('none').apply(cleaner.clean_text)

# 超参数调优并训练最优模型
model, tokenizer = train_final_bert_model(data, model_name=LOCAL_BERT_PATH, epochs=4, batch_size=32, max_len=128, learning_rate=2e-5)

In [ ]:
def predict_single_sample_bert(model, tokenizer, text, keyword,location, max_len=128):
    """预测单条样本"""
    cleaner = CustomTweetCleaner()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # print(device)
    
    # # 清理样本
    # cleaned_text = cleaner.clean_text(text)
    # cleaned_keyword = cleaner.clean_text(keyword if keyword else 'none')
    
    # 拼接keyword和text
    input_text = f"{location} [SEP] {keyword} [SEP] {text}"
    
    # 编码输入
    encoding = tokenizer.encode_plus(
        input_text,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # 预测
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        prediction = torch.argmax(outputs.logits, dim=1).cpu().numpy()[0]
    
    return prediction

In [ ]:
def predict_test_set(model, tokenizer, test_data, max_len=128):
    """预测test.csv并生成Kaggle提交文件"""
    cleaner = CustomTweetCleaner()
    
    # 清理测试数据
    test_data['text_cleaned'] = test_data['text'].apply(cleaner.clean_text)
    test_data['keyword'] = test_data['keyword'].fillna('none').apply(cleaner.clean_text)
    test_data['location'] = test_data['location'].fillna('none').apply(cleaner.clean_text)
    
    # 预测
    predictions = []
    for _, row in tqdm(test_data.iterrows(), total=len(test_data), desc="Predicting test set"):
        pred = predict_single_sample_bert(model, tokenizer, row['text'], row['keyword'], max_len)
        predictions.append(pred)
    
    # 生成提交文件
    submission = pd.DataFrame({'id': test_data['id'], 'target': predictions})
    #submission.to_csv('submission.csv', index=False)
    #print("Submission file saved as 'submission.csv'")
    
    return submission

In [ ]:
test_data = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
submission = predict_test_set(model, tokenizer, test_data)

In [ ]:
submission.to_csv("submission.csv", index=False)
print("Your submission was successfully saved!")

In [ ]:
# sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
# sample_submission.head()

In [ ]:
# sample_submission["target"] = np.argmax(classifier.predict(X_test), axis=1)
# sample_submission.to_csv("submission.csv", index=False)
# print("Your submission was successfully saved!")
# submission = predict_test_set(model, tokenizer, test_data)